In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import umap
import os
from tqdm import tqdm

def plot_top_journal_clusters(csv_path, save_dir):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    # 1. 读取 CSV 索引
    df = pd.read_csv(csv_path)
    
    # 2. 读取 .pt 特征文件
    features = []
    for path in tqdm(df['feature_path'], desc="[Step 1/5] 读取特征文件"):
        # 加载 tensor 并转为 numpy 数组
        feat = torch.load(path, map_location='cpu')
        features.append(feat.detach().numpy().flatten())
    X = np.array(features)

    # 3. 确定预测类别 (无监督聚类逻辑)
    # 根据相似度得分判定：得分高的即为预测簇
    df['pred_cluster'] = np.where(
        df['similarity_to_class_0_average'] > df['similarity_to_class_1_average'], 0, 1
    ).astype(int)

    # 4. 执行降维
    print("\n[Step 2/5] 启动降维引擎...")
    
    # print("▶ 正在运行 PCA (线性映射)...")
    # X_pca = PCA(n_components=2).fit_transform(X)
    # 
    print("▶ 正在运行 UMAP (拓扑结构)...")
    X_umap = umap.UMAP(
        n_neighbors=15, 
        min_dist=0.1, 
        random_state=42,
        verbose=False 
    ).fit_transform(X)

    # 5. 绘图渲染
    print("\n[Step 3/5] 渲染顶刊风格图表...")
    methods = [('UMAP', X_umap)]
    
    ##     methods = [('PCA', X_pca), ('UMAP', X_umap)]选择方法
    fig, axes = plt.subplots(1, 1, figsize=(18, 10), dpi=300)
    
    # 设置顶刊常用的配色方案
    # 0类: 浅蓝色 (#4DBBD5FF), 1类: 珊瑚红 (#E64B35FF)
    palette = {0: "#4DBBD5FF", 1: "#E64B35FF"} 
    
    # 设置形状映射
    # 0类: 圆圈 ('o'), 1类: 大叉 ('X')
    marker_map = {0: "o", 1: "X"} 
    common_order = [0, 1]
    if not isinstance(axes, (list, np.ndarray)):
        axes = [axes]  # 包装成列表
    for ax, (name, coords) in tqdm(zip(axes, methods), total=1, desc="[Step 4/5] 渲染子图"):
        sns.scatterplot(
            x=coords[:, 0], 
            y=coords[:, 1], 
            hue=df['pred_cluster'],    # 颜色绑定预测簇
            style=df['pred_cluster'],  # 形状也绑定预测簇 (实现联动)
            hue_order=common_order,   
            style_order=common_order, 
            markers=marker_map,       
            palette=palette, 
            s=160,                     # 增大点的大小使叉号更明显
            alpha=0.8, 
            ax=ax, 
            edgecolor='w', 
            linewidth=0.8
        )
        
        # 细节优化
        ax.set_title(f"{name} Analysis", fontsize=15, fontweight='bold', pad=15)
        ax.set_xlabel("Feature 1", fontsize=12)
        ax.set_ylabel("Feature 2", fontsize=12)
        
        # 优化图例：只显示预测簇的分类
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=['Cluster 0', 'Cluster 1'], 
                  title="Predicted Group", loc='best', frameon=True)

    plt.suptitle(f"Feature Space Visualization (Total Samples N=104783", 
                 fontsize=18, y=1.02, fontweight='bold')
    
    plt.tight_layout()
    
    # 保存结果
    save_path = os.path.join(save_dir, "cluster_visualization_final.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    
    print(f"\n[Step 5/5] ✅ 任务成功完成！")
    print(f"📊 统计：Cluster 0 样本数: {sum(df['pred_cluster']==0)}")
    print(f"📊 统计：Cluster 1 样本数: {sum(df['pred_cluster']==1)}")
    print(f"💾 文件保存至: {save_path}")

# --- 执行脚本 ---
# 请确保您的 CSV 路径和保存目录路径正确
csv_file = '/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/pretrain_class_sim.csv'
output_dir = '/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/聚类可视化1'

plot_top_journal_clusters(csv_file, output_dir)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import umap
import os
from tqdm import tqdm

def plot_top_journal_clusters(csv_path, save_dir):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    # 1. 读取 CSV 索引
    df = pd.read_csv(csv_path)
    
    # 2. 读取 .pt 特征文件
    features = []
    for path in tqdm(df['feature_path'], desc="[Step 1/5] 读取特征文件"):
        # 加载 tensor 并转为 numpy 数组
        feat = torch.load(path, map_location='cpu')
        features.append(feat.detach().numpy().flatten())
    X = np.array(features)

    # 3. 确定预测类别 (无监督聚类逻辑)
    # 根据相似度得分判定：得分高的即为预测簇
    df['pred_cluster'] = df['label'].astype(int)

    # 4. 执行降维
    print("\n[Step 2/5] 启动降维引擎...")
    
    # print("▶ 正在运行 PCA (线性映射)...")
    # X_pca = PCA(n_components=2).fit_transform(X)
    # 
    print("▶ 正在运行 UMAP (拓扑结构)...")
    X_umap = umap.UMAP(
        n_neighbors=15, 
        min_dist=0.1, 
        random_state=42,
        verbose=False 
    ).fit_transform(X)

    # 5. 绘图渲染
    print("\n[Step 3/5] 渲染顶刊风格图表...")
    methods = [('UMAP', X_umap)]
    fig, axes = plt.subplots(1, 1, figsize=(18, 10), dpi=300)
    
    # 设置顶刊常用的配色方案
    # 0类: 浅蓝色 (#4DBBD5FF), 1类: 珊瑚红 (#E64B35FF)
    palette = {0: "#4DBBD5FF", 1: "#E64B35FF"} 
    
    # 设置形状映射
    # 0类: 圆圈 ('o'), 1类: 大叉 ('X')
    marker_map = {0: "o", 1: "X"} 
    common_order = [0, 1]
    if not isinstance(axes, (list, np.ndarray)):
        axes = [axes]  # 包装成列表
    for ax, (name, coords) in tqdm(zip(axes, methods), total=1, desc="[Step 4/5] 渲染子图"):
        sns.scatterplot(
            x=coords[:, 0], 
            y=coords[:, 1], 
            hue=df['pred_cluster'],    # 颜色绑定预测簇
            style=df['pred_cluster'],  # 形状也绑定预测簇 (实现联动)
            hue_order=common_order,   
            style_order=common_order, 
            markers=marker_map,       
            palette=palette, 
            s=160,                     # 增大点的大小使叉号更明显
            alpha=0.8, 
            ax=ax, 
            edgecolor='w', 
            linewidth=0.8
        )
        
        # 细节优化
        ax.set_title(f"{name} Analysis", fontsize=15, fontweight='bold', pad=15)
        ax.set_xlabel("Feature 1", fontsize=12)
        ax.set_ylabel("Feature 2", fontsize=12)
        
        # 优化图例：只显示预测簇的分类
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=['Cluster 0', 'Cluster 1'], 
                  title="Predicted Group", loc='best', frameon=True)

    plt.suptitle(f"Feature Space Visualization (Total Samples N={len(df)})", 
                 fontsize=18, y=1.02, fontweight='bold')
    
    plt.tight_layout()
    
    # 保存结果
    save_path = os.path.join(save_dir, "cluster_visualization_final_1.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    
    print(f"\n[Step 5/5] ✅ 任务成功完成！")
    print(f"📊 统计：Cluster 0 样本数: {sum(df['pred_cluster']==0)}")
    print(f"📊 统计：Cluster 1 样本数: {sum(df['pred_cluster']==1)}")
    print(f"💾 文件保存至: {save_path}")

# --- 执行脚本 ---
# 请确保您的 CSV 路径和保存目录路径正确
csv_file = '/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/inference_distance_samples.csv'
output_dir = '/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/聚类可视化1'

plot_top_journal_clusters(csv_file, output_dir)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import umap
import os
from tqdm import tqdm

def plot_top_journal_clusters(csv_path, save_dir):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    # 1. 读取 CSV 索引
    df = pd.read_csv(csv_path)
    
    # 2. 读取 .pt 特征文件
    features = []
    for path in tqdm(df['feature_path'], desc="[Step 1/5] 读取特征文件"):
        # 加载 tensor 并转为 numpy 数组
        feat = torch.load(path, map_location='cpu')
        features.append(feat.detach().numpy().flatten())
    X = np.array(features)

    # 3. 确定预测类别 (无监督聚类逻辑)
    # 根据相似度得分判定：得分高的即为预测簇
    df['pred_cluster'] = df['label'].astype(int)

    # 4. 执行降维
    print("\n[Step 2/5] 启动降维引擎...")
    
    # print("▶ 正在运行 PCA (线性映射)...")
    # X_pca = PCA(n_components=2).fit_transform(X)
    # 
    print("▶ 正在运行 UMAP (拓扑结构)...")
    X_umap = umap.UMAP(
        n_neighbors=15, 
        min_dist=0.1, 
        random_state=42,
        verbose=False 
    ).fit_transform(X)

    # 5. 绘图渲染
    print("\n[Step 3/5] 渲染顶刊风格图表...")
    methods = [('UMAP', X_umap)]
    fig, axes = plt.subplots(1, 1, figsize=(18, 10), dpi=300)
    if not isinstance(axes, (list, np.ndarray)):
        axes = [axes]
    # 设置顶刊常用的配色方案
    # 0类: 浅蓝色 (#4DBBD5FF), 1类: 珊瑚红 (#E64B35FF)
    palette = {0: "#4DBBD5FF", 1: "#E64B35FF"} 
    
    # 设置形状映射
    # 0类: 圆圈 ('o'), 1类: 大叉 ('X')
    marker_map = {0: "o", 1: "X"} 
    common_order = [0, 1]
    
    for ax, (name, coords) in tqdm(zip(axes, methods), total=1, desc="[Step 4/5] 渲染子图"):
        sns.scatterplot(
            x=coords[:, 0], 
            y=coords[:, 1], 
            hue=df['pred_cluster'],    # 颜色绑定预测簇
            style=df['pred_cluster'],  # 形状也绑定预测簇 (实现联动)
            hue_order=common_order,   
            style_order=common_order, 
            markers=marker_map,       
            palette=palette, 
            s=160,                     # 增大点的大小使叉号更明显
            alpha=0.8, 
            ax=ax, 
            edgecolor='w', 
            linewidth=0.8
        )
        
        # 细节优化
        ax.set_title(f"{name} Analysis", fontsize=15, fontweight='bold', pad=15)
        ax.set_xlabel("Feature 1", fontsize=12)
        ax.set_ylabel("Feature 2", fontsize=12)
        
        # 优化图例：只显示预测簇的分类
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=['Cluster 0', 'Cluster 1'], 
                  title="Predicted Group", loc='best', frameon=True)

    plt.suptitle(f"Feature Space Visualization (Total Samples N={len(df)})", 
                 fontsize=18, y=1.02, fontweight='bold')
    
    plt.tight_layout()
    
    # 保存结果
    save_path = os.path.join(save_dir, "cluster_visualization_final_2.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    
    print(f"\n[Step 5/5] ✅ 任务成功完成！")
    print(f"📊 统计：Cluster 0 样本数: {sum(df['pred_cluster']==0)}")
    print(f"📊 统计：Cluster 1 样本数: {sum(df['pred_cluster']==1)}")
    print(f"💾 文件保存至: {save_path}")

# --- 执行脚本 ---
# 请确保您的 CSV 路径和保存目录路径正确
csv_file = '/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/inference_distance_samples_filtered.csv'
output_dir = '/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/聚类可视化1'

plot_top_journal_clusters(csv_file, output_dir)